# Import Statements

In [ ]:
import os
import custom_cmap
from PIL import Image
from IPython.display import display

import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd

from moria import reduce
from pathlib import Path

from astropy.visualization import PercentileInterval
from astropy.io import fits
from astropy.visualization import LogStretch, ImageNormalize
import plotly.express as px
import numpy as np
from astropy.visualization import ImageNormalize, AsinhStretch, SqrtStretch, LogStretch, PowerStretch

plt.rcParams.update({'font.size':25})
plt.rc('text', usetex=True)
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

# Understanding the main directory. 

The directory structure for your data analysis should be organized as follows. 

<li>00.DATA</li>
<li>01.XYM</li>
<li>02.CMD</li>
<li>03.LOC_TRANS</li>
<li>04.PSF_EXTRACT</li>
<li>05.COORD_TRANS(OPTIONAL)</li>
<li>06.FIT</li>
<li>07.CALIBRATION</li>

This directory structure has been set up in "MORIA/data". 

All necessary scripts are included within the corresponding folders under MORIA/data. Therefore, the simplest way to run MORIA on your target is to copy all eight folders from MORIA/data to the location where you intend to perform your analysis. Begin by placing your exposures in data/00.DATA.

After you finish running this notebook, you will be able to use the "_flc" exposures to generate an output image stack in the F814W and F606W HST filters.

NOTE: chmod +x program.src is a useful command to use whenever permissions are denied for a script. If any of these scripts fail, each folder in the list above (00.DATA, 01.XYM, 02.CMD, etc.) has an output "log file" for the Fortran scripts we run. If the "log file" for the step you ran shows a "permission denied" error, it is very likely that you need to run chmod _x program.src for each ".src" file used in this pipeline.

In [ ]:
#This is the directory where you are processing your data. This does not point at MORIA.
directory = os.getcwd()

# Data Preparation 

When you installed MORIA, several fortran scripts were compiled directly in "MORIA/src/fortran_compile". This step copies the fortran files into the correct directories. 

In [ ]:
reduce.data_prep_early(directory)

# STEP 1

<li>Download the _flc files. Place them appropriately in the F814W and F606W directories in 00.DATA.</li>
<li> The cell below will convert the _flc files to _WJ2 files using run_convert_C1K1C.src</li>

These "_flc" exposures include the charge-transfer efficiency (CTE) corrections. MORIA first converts these "_flc" exposures into "_WJ2" files to convert to a full-chip co-ordiante system that are placed into appropriate reference geometry for subsequent steps. 

In [ ]:
reduce.run_xgf_conversion(directory)



# Step 2 

Prepare the IN.* files taken in by the fortran scripts in 01.XYM and 02.CMD directories. 

This step is important since the IN.* files in each directory are used to provide input to the backend Fortran scripts.

In [ ]:
reduce.data_prep(directory)

# Create MATCHUP files

In the cell below, we will achieve the following objectives:

1. Use the corresponding PSF files for each filter to create an .XYM file for every exposure. A library PSF model is used to generate the .XYM files, which contain the stellar positions and magnitudes for each HST image exposure. 
2. Generate the output file TRANS.xym2mat, along with 16 MAT.0 files. Note that the first line of IN.xym2mat and files beginning with 00 only define the reference coordinates used for the output. Therefore, the 00 file does not need to be a *_WJ2.xym file.
3. Find all the objects that could be found in at least 80% of the exposures for each filter. 

In [ ]:
reduce.matchup_files(directory)

Once the cell above finishes running, you should have two MATCHUP files. One is stored in 01.XYM/F814W for the F814W filter and the other in 01.XYM/F606W for the F606W filter. 

We will open the MATCHUP files to inspect it further. The MATCHUP file we will open will be for the F814W folder. You can uncomment the filename_606W line in the cell below to look at the F606W MATCHUP file.

In [ ]:
filename_814W = Path(directory).resolve()/f"01.XYM/F814W/MATCHUP.F814W.XYM.02"
filename_606W = Path(directory).resolve()/f"01.XYM/F606W/MATCHUP.F606.XYM" 

cols = ["xbar", "ybar", "mbar", "xsig", "ysig", "msig", "qbar", "Nf", "Ng", "Nm", "Nmin", "Nstar", "pki", "pkj", "pkp", "pkn", "pku"]
df = pd.read_csv(filename_814W, sep=r"\s+", comment="#", header=None, names=cols)

In [ ]:
df

From the MATCHUP file opened above, here are the most important things to note:

1. xbar and ybar correspond to the mean x and y star positions (in pixel coordinates) for stars in the HST images.
2. mbar is the mean F814W I magnitude of the stars.
4. xsig, ysig, msig are the errors on positions and magnitudes.

Using this information from the MATCHUP files, we are ready to create a stack of the HST exposures. 

# Step 3

This step will create a file called "outpuq.fits", which is a stack of your HST exposures. A stack will be created in both filters (F814W and F606W) and you can use it to identify your target.

In [ ]:
reduce.run_output_stack(directory)

You are now ready to view the output stacks for both filters using DS9.

Alternatively, you can view this output stack in the jupyter notebook by running the cell below. The output stack for the F814W filter is kept in 01.XYM/F814W and the output stack for the F606W filter is kept in 01.XYM/F606W. We open the output stack for F814W below. You can uncomment the filename_606W line in the cell below to look at the F606W output stack.

The scaling can be adjusted depending on your preference by changing the argument given to "ImageNormalize" in the cell below. 

In [ ]:
##############
# READ FILES #
##############

filename_814W = Path(directory).resolve()/f"01.XYM/F814W/outputq_F814W.fits"
#filename_606W = Path(directory).resolve()/f"01.XYM/F814W/outputq_F606W.fits"

##################
# OPEN FITS FILE #
##################
hdul = fits.open(filename_814W)
data = hdul[-1].data
hdul.close()
data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)


##########################
# SCALE THE OUTPUT STACK #
##########################
norm = ImageNormalize(data, stretch=AsinhStretch(0.02))
scaled = norm(data)
nonbinary_colors = custom_cmap.nb_colors()

########
# PLOT #
########
fig = px.imshow(scaled, origin='lower', color_continuous_scale=nonbinary_colors, title="Output Stack (F814W)", aspect='equal')
fig.update_layout(width=700, height=700, coloraxis_colorbar=dict(title="Log Intensity", tickvals=[]))
fig.show()
